## Imports

In [ ]:
import requests
import csv
import os
import re
import io
import calendar
from PyPDF2 import PdfReader
from datetime import datetime

## File Paths, Downloading PDFs, and Helper Functions for Data Extraction

In [ ]:


# Base data path using current notebook directory
notebook_dir = os.getcwd()
DATE_PATTERN = re.compile(
    r"^(\d{2})-(\d{2})-(\d{4})(?:_\d+)?$",
    re.IGNORECASE
)
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')

# Browser-like headers to avoid the Lawrence site blocking Python requests
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,text/html,*/*",
    "Referer": "https://www.lawpd.com/",
}

def get_filename_from_content_disposition(header):
    if "filename=" in header.lower():
        return header.split("filename=")[-1].strip('"; ')
    return None

def parse_date_from_filename(filename):
    base_name = os.path.splitext(filename)[0]
    match = DATE_PATTERN.match(base_name)

    if match:
        mm, dd, yyyy = int(match.group(1)), int(match.group(2)), int(match.group(3))
        return yyyy, mm, dd

    return None, None, None
def sanitize_filename(name):
    """
    Remove characters that are illegal or awkward in filenames.
    """
    safe = re.sub(r"[^a-zA-Z0-9_\-. ]+", "_", name)
    return safe[:100]
os.makedirs(DATA_DIR, exist_ok=True) #tracks and lists failures in csv

def download_pdfs(start_id=49804, end_id=51475):   ## INPUT THE ID DATES for what you are trying to extract : )
    os.makedirs(DATA_DIR, exist_ok=True)
    failure_csv_path = os.path.join(DATA_DIR, "failures.csv")
    failure_file = open(
    failure_csv_path,
    mode="w",
    newline="",
    encoding="utf-8")
    failure_writer = csv.writer(failure_file)
    failure_writer.writerow(["Document ID", "Reason"])
    
    

    total = end_id - start_id + 1

    skipped = {
        "http": 0,
        "not_pdf": 0,
        "missing_filename": 0,
        "wrong_year": 0,
        "error": 0
    }

    saved = 0

    for count, doc_id in enumerate(range(start_id, end_id + 1), start=1):
        if count % 500 == 0:
            print(f"📦 Progress: {count} / {total} checked")

        url = f"https://www.lawpd.com/DocumentCenter/View/{doc_id}"

        try:
            res = requests.get(
                url,
                headers=HEADERS,
                timeout=20,
                allow_redirects=True
            )

            if res.status_code != 200:
                skipped["http"] += 1
                failure_writer.writerow([doc_id, f"HTTP {res.status_code}"])
                continue

            content_type = res.headers.get("Content-Type", "").lower()
            if "pdf" not in content_type:
                skipped["not_pdf"] += 1
                failure_writer.writerow(
                    [doc_id, f"Not PDF (Content-Type: {content_type})"]
                )
                continue

            filename = get_filename_from_content_disposition(
                res.headers.get("Content-Disposition", "")
            )

            if not filename:
                skipped["missing_filename"] += 1
                failure_writer.writerow([doc_id, "Missing filename"])
                continue

            if not filename.lower().endswith(".pdf"):
                filename += ".pdf"
            filename = re.sub(
                r"_(\d+)(\.pdf)$",
                r"\2",
                filename,
                flags=re.IGNORECASE
            )
            filename = sanitize_filename(filename)

            year, month, day = parse_date_from_filename(filename)

            if year is None:
                skipped["wrong_year"] += 1
                failure_writer.writerow([doc_id, f"Invalid date filename: {filename}"])
                continue

            # Build folder path: .../data/2023_january
            month_folder = f"{year}_{calendar.month_name[month].lower()}"
            save_dir = os.path.join(DATA_DIR, month_folder)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(save_dir, filename)

            with open(save_path, "wb") as f:
                f.write(res.content)

            saved += 1
            print(f"[{count}/{total}] ✅ Saved: {filename}")

        except Exception as e:
            skipped["error"] += 1
            failure_writer.writerow(
                [doc_id, f"{type(e).__name__}: {e}"]
            )
            if skipped["error"] <= 5:
                print(f"{doc_id}: {type(e).__name__}: {e}")
    failure_file.close()
    print(f"Done. Saved {saved} PDFs to {DATA_DIR}")
    print(f"Skipped: {skipped}")

In [ ]:
download_pdfs()